# Cohere Transcribe Arabic (2B) — LoRA fine-tuning

Sibling of `asr_qwen3_finetune.ipynb` (Qwen3-ASR) and `asr_conformer_ctc_finetune.ipynb`
(FastConformer-CTC): **same pipeline shape** (ConfigAPI → ModelAdapter → PredictAPI →
EvaluateAPI → TrainAPI) but for **`CohereLabs/cohere-transcribe-arabic-07-2026`**.

### Model facts
* 2B-param **Conformer encoder + Transformer decoder** seq2seq, transformers-native
  (`CohereAsrForConditionalGeneration`, needs transformers >= 5.4; this env has 5.14.1).
* Arabic + English + code-switch; 16 kHz mono log-mel input.
* **GATED repo** (auto-approved after accepting terms): downloading weights requires
  `HF_TOKEN` / `hf auth login` with an account that accepted the license on the model page.

### Key differences from the Qwen3-ASR notebook
1. **Prompted seq2seq decoder, not chat.** The processor builds a 10-token decoder prompt
   (`<|startofcontext|><|startoftranscript|>...<|ar|><|pnc|>...`) via
   `processor.get_decoder_prompt_ids(language, punctuation)`. Training uses explicit
   teacher forcing: `decoder_input_ids = prompt + transcript`, `labels` = same sequence
   shifted left with the prompt positions masked to `-100`, `loss = model(**batch).loss`.
2. **Inference** is `processor(audio, language="ar", sampling_rate=16000)` →
   `model.generate` → strip the prompt → `tokenizer.batch_decode(skip_special_tokens=True)`
   (with chunk reassembly if the extractor split clips longer than `max_audio_clip_s`).
3. **Real PEFT LoRA** with **dynamically discovered** `nn.Linear` targets (encoder/decoder
   projection leaf names are printed at apply time), since CohereAsr module naming is new.
4. **Same venv as Qwen3** (`/workspace/venv_qwen_gpu`, transformers 5.14.1 + torch cu128).

## Cell 1 — Environment

In [1]:
# Env: /workspace/venv_qwen_gpu (transformers 5.14.1 + peft + torch cu128). To rebuild:
# !pip install "transformers>=5.13" peft accelerate datasets jiwer soundfile librosa bitsandbytes mlflow pandas
import os, json, gc, math, time, random, hashlib, warnings, shutil
from pathlib import Path
from dataclasses import dataclass, field, asdict, replace
from typing import Any, Dict, List, Optional, Callable

import numpy as np, torch, torch.nn as nn
warnings.filterwarnings("ignore")
os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "0")
print(torch.__version__, torch.cuda.is_available())

2.8.0+cu128 True


## Cell 2 — Config knobs

In [2]:
MODEL_NAME = "CohereLabs/cohere-transcribe-arabic-07-2026"

LANG          = "ar"                # CohereAsr decoder-prompt language code
SMOKE_TEST    = True                # tiny subsets + 2 epochs
SEED          = 42

DEVICE        = "cuda" if torch.cuda.is_available() else "cpu"
COMPUTE_DTYPE = torch.bfloat16 if DEVICE == "cuda" else torch.float32

ROOT          = Path(os.environ.get("ASR_ENV_ROOT", "/workspace/asr_env"))
MODEL_CACHE   = ROOT / "models"
PRED_DIR      = ROOT / "preds"
METRIC_DIR    = ROOT / "metrics"
CKPT_DIR      = ROOT / "checkpoints"
for d in (MODEL_CACHE, PRED_DIR, METRIC_DIR, CKPT_DIR): d.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(MODEL_CACHE / "hf")
import mlflow
# mlflow 3.x deprecated the plain "file:" tracking backend (raises unless
# MLFLOW_ALLOW_FILE_STORE=true) -- sqlite is the currently-recommended local backend and
# also lets `mlflow ui --backend-store-uri ...` browse runs later. Artifact location is
# set explicitly on first creation since sqlite backends don't default one on their own.
MLFLOW_TRACKING_URI = os.environ.get("MLFLOW_TRACKING_URI", f"sqlite:///{ROOT / 'mlflow.db'}")
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
_MLFLOW_EXPERIMENT = "arabic-asr-cohere-transcribe"
if mlflow.get_experiment_by_name(_MLFLOW_EXPERIMENT) is None:
    mlflow.create_experiment(_MLFLOW_EXPERIMENT, artifact_location=f"file:{ROOT / 'mlruns' / _MLFLOW_EXPERIMENT}")
mlflow.set_experiment(_MLFLOW_EXPERIMENT)

# The repo is GATED: verify credentials up front so the failure mode is obvious.
from huggingface_hub import get_token
_tok = get_token() or os.environ.get("HF_TOKEN")
if not _tok:
    print("[WARN] No Hugging Face token found (hf auth login / HF_TOKEN). "
          f"{MODEL_NAME} is a gated repo -- the model download in Cell 11 will fail with 401 "
          "until a token whose account accepted the license is available.")

def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed()
print(f"DEVICE={DEVICE} | dtype={COMPUTE_DTYPE} | ROOT={ROOT}")

DEVICE=cuda | dtype=torch.bfloat16 | ROOT=/workspace/asr_env


## Cell 3 — Arabic normalization + WER/CER (identical to the Qwen3/Conformer notebooks)

In [3]:
import re, unicodedata, jiwer

_DIAC = re.compile(r"[\u0617-\u061A\u064B-\u0652\u0640]")
_PUNC = re.compile(r"[^\w\s\u0621-\u064A]")

def normalize_ar(t: str) -> str:
    """Diacritic strip, tatweel removal, alef/ya/ta-marbuta unification."""
    if t is None: return ""
    t = unicodedata.normalize("NFKC", str(t))
    t = _DIAC.sub("", t)
    t = re.sub("[\u0622\u0623\u0625\u0671]", "\u0627", t)   # alef variants -> alef
    t = t.replace("\u0649", "\u064A")                          # alef maqsura -> ya
    t = t.replace("\u0629", "\u0647")                          # ta marbuta -> ha
    t = t.replace("\u0624", "\u0648").replace("\u0626", "\u064A")
    t = _PUNC.sub(" ", t)
    return re.sub(r"\s+", " ", t).strip()

def compute_wer_cer(preds, refs, normalize=True):
    if normalize:
        preds = [normalize_ar(p) for p in preds]
        refs  = [normalize_ar(r) for r in refs]
    keep = [(p, r) for p, r in zip(preds, refs) if r.strip()]
    if not keep: return {"wer": float("nan"), "cer": float("nan"), "n": 0}
    p, r = zip(*keep)
    return {"wer": jiwer.wer(list(r), list(p)),
            "cer": jiwer.cer(list(r), list(p)),
            "n": len(r)}

## Cell 4 — ConfigAPI

`target_modules=None` means **discover at apply time**: CohereAsr's projection leaf names are
new, so `apply_lora` scans `named_modules()` for `nn.Linear` leaves matching the usual
attention/FF naming and prints what it picked. 2B params on a 24 GB card -> train batch 1 +
grad-accum 8, eval batch 2. `max_audio_seconds=30` matches the smoke data (the extractor
chunks clips longer than its `max_audio_clip_s` internally; we keep clips below it).

In [4]:
@dataclass
class LoRAConfigSpec:
    r: int = 32
    lora_alpha: int = 32
    lora_dropout: float = 0.05
    bias: str = "none"
    target_modules: Optional[List[str]] = None   # None -> discovered in apply_lora, printed
    modules_to_save: Optional[List[str]] = None
    task_type: Optional[str] = None      # custom seq2seq head -> leave None (LoRA still injects)

@dataclass
class TrainConfigSpec:
    num_epochs: int = 50
    early_stopping_patience: int = 3
    metric_for_best: str = "wer"
    greater_is_better: bool = False
    per_device_train_batch_size: int = 1   # 2B model
    per_device_eval_batch_size: int = 2
    gradient_accumulation_steps: int = 8
    learning_rate: float = 1e-4
    warmup_ratio: float = 0.05
    weight_decay: float = 0.0
    max_grad_norm: float = 1.0
    optim: str = "adamw_bnb_8bit"
    bf16: bool = True
    gradient_checkpointing: bool = False   # not wired (PEFT+grad-ckpt needs enable_input_require_grads)
    dataloader_num_workers: int = 0        # 0 avoids fork+CUDA issues when collate runs the processor
    max_audio_seconds: float = 30.0
    max_label_tokens: int = 256
    save_total_limit: int = 3
    save_steps: Optional[int] = None      # None -> auto: max(200, steps_per_epoch // 3)
    dataloader_pin_memory: bool = True
    dataloader_persistent_workers: bool = True
    dataloader_prefetch_factor: int = 4

class ConfigAPI:
    _LORA = {MODEL_NAME: LoRAConfigSpec()}
    _TRAIN = {MODEL_NAME: TrainConfigSpec()}
    @classmethod
    def lora(cls, name)  -> LoRAConfigSpec:  return cls._LORA.get(name, LoRAConfigSpec())
    @classmethod
    def train(cls, name) -> TrainConfigSpec: return cls._TRAIN.get(name, TrainConfigSpec())

print(json.dumps(asdict(ConfigAPI.lora(MODEL_NAME)), indent=2))
print(json.dumps(asdict(ConfigAPI.train(MODEL_NAME)), indent=2))

{
  "r": 32,
  "lora_alpha": 32,
  "lora_dropout": 0.05,
  "bias": "none",
  "target_modules": null,
  "modules_to_save": null,
  "task_type": null
}
{
  "num_epochs": 50,
  "early_stopping_patience": 3,
  "metric_for_best": "wer",
  "greater_is_better": false,
  "per_device_train_batch_size": 1,
  "per_device_eval_batch_size": 2,
  "gradient_accumulation_steps": 8,
  "learning_rate": 0.0001,
  "warmup_ratio": 0.05,
  "weight_decay": 0.0,
  "max_grad_norm": 1.0,
  "optim": "adamw_bnb_8bit",
  "bf16": true,
  "gradient_checkpointing": false,
  "dataloader_num_workers": 0,
  "max_audio_seconds": 30.0,
  "max_label_tokens": 256,
  "save_total_limit": 3,
  "save_steps": null,
  "dataloader_pin_memory": true,
  "dataloader_persistent_workers": true,
  "dataloader_prefetch_factor": 4
}


## Cell 5 — ModelAdapter ABC

In [5]:
from abc import ABC, abstractmethod

class ModelAdapter(ABC):
    name: str
    loss_type: str = "seq2seq"
    supports_unsloth: bool = False

    def __init__(self, model_name: str, lang: str = LANG):
        self.model_name = model_name; self.lang = lang
        self.model = None; self.processor = None

    @abstractmethod
    def load_base(self): ...
    @abstractmethod
    def preprocess(self, example: Dict) -> Dict: ...
    @abstractmethod
    def collate(self, features: List[Dict]) -> Dict[str, Any]: ...
    @abstractmethod
    def generate(self, batch: Dict) -> List[str]: ...

    def _discover_lora_targets(self) -> List[str]:
        """Leaf names of nn.Linear modules that look like attention/FF projections."""
        # Attention CONTENT projections + feed-forward only -- matching the FastConformer
        # sibling notebook. Deliberately excluded: `relative_k_proj` (projects positional
        # embeddings, not content), bare `linear` (the conv-subsampling frontend), and the
        # output heads.
        pat = re.compile(r"^(q|k|v|o)_proj$|^(gate|up|down)_proj$"
                         r"|^linear_(q|k|v|out)$|^linear[12]$|^fc[12]$|^w[123]$")
        names = {n.split(".")[-1] for n, m in self.model.named_modules() if isinstance(m, nn.Linear)}
        targets = sorted(n for n in names if pat.match(n) and n not in {"lm_head", "proj_out"})
        if not targets:
            raise RuntimeError(f"No LoRA targets matched. Available Linear leaves: {sorted(names)}")
        return targets

    def apply_lora(self, spec: "LoRAConfigSpec"):
        """PEFT LoRA. CohereAsr projections are torch.nn.Linear, so this Just Works."""
        from peft import LoraConfig, get_peft_model
        if spec.target_modules is None:
            spec = replace(spec, target_modules=self._discover_lora_targets())
            print(f"[lora] discovered target_modules: {spec.target_modules}")
        kw = dict(r=spec.r, lora_alpha=spec.lora_alpha, lora_dropout=spec.lora_dropout,
                  bias=spec.bias, target_modules=spec.target_modules)
        if spec.modules_to_save: kw["modules_to_save"] = spec.modules_to_save
        if spec.task_type:       kw["task_type"] = spec.task_type
        self.model = get_peft_model(self.model, LoraConfig(**kw))
        self.model.print_trainable_parameters()
        print("[lora] peft"); return self.model

    def trainable_parameters(self) -> List["torch.nn.Parameter"]:
        return [p for p in self.model.parameters() if p.requires_grad]

    def save_checkpoint(self, dir_):
        """Default: self.model IS the PEFT-wrapped module. Concrete adapters whose LoRA
        lives elsewhere (self.peft, manual injection) override this."""
        Path(dir_).mkdir(parents=True, exist_ok=True)
        self.model.save_pretrained(str(dir_))

    def load_checkpoint(self, dir_):
        from safetensors.torch import load_file
        from peft import set_peft_model_state_dict
        sd = load_file(str(Path(dir_) / "adapter_model.safetensors"))
        set_peft_model_state_dict(self.model, sd)


## Cell 6 — CohereAsr adapter

Teacher-forcing detail: `CohereAsrForConditionalGeneration.forward` computes
`loss = loss_function(logits, labels)` **without an internal shift** when
`decoder_input_ids` is passed explicitly (it only builds `decoder_input_ids` by
right-shifting `labels` when you omit them — which would drop the language/task prompt).
So the collate builds `full = prompt + transcript + [eos]`, feeds
`decoder_input_ids = full[:-1]`, `labels = full[1:]` with the prompt positions masked to
`-100`: logits at position *t* predict token *t+1*, and only transcript+eos count in the loss —
exactly matching the `generate` path, which starts from the same 10-token prompt.

In [6]:
class CohereAsrAdapter(ModelAdapter):
    """CohereLabs/cohere-transcribe-arabic-07-2026 (transformers-native CohereAsr).
      * load:   AutoProcessor + CohereAsrForConditionalGeneration (bf16)  [GATED repo]
      * TRAIN:  explicit teacher forcing (see markdown above) -> loss = model(**in).loss
      * INFER:  processor(audio, language, sampling_rate=16000) -> model.generate
                -> strip prompt -> tokenizer.batch_decode(skip_special_tokens=True)
                (+ processor._reassemble_chunk_texts when clips got chunked)
      * inputs MUST be cast to the model dtype (BatchFeature.to(device, dtype) casts float only).
      * LoRA:   real PEFT on discovered encoder/decoder nn.Linear projections.
    """
    loss_type = "seq2seq"
    supports_unsloth = False

    def load_base(self):
        from transformers import AutoProcessor, CohereAsrForConditionalGeneration
        print(f"[load] {self.model_name}")
        self.processor = AutoProcessor.from_pretrained(self.model_name)
        self.model = CohereAsrForConditionalGeneration.from_pretrained(
            self.model_name, dtype=COMPUTE_DTYPE).to(DEVICE)
        self.model.config.use_cache = False
        tok = self.processor.tokenizer
        self._prompt_ids = list(self.processor.get_decoder_prompt_ids(
            language=self.lang, punctuation=True))
        eos = self.model.generation_config.eos_token_id
        if isinstance(eos, (list, tuple)): eos = eos[0]
        self._eos_id = int(eos if eos is not None else tok.eos_token_id)
        pad = self.model.config.pad_token_id
        if pad is None: pad = tok.pad_token_id if tok.pad_token_id is not None else self._eos_id
        self._pad_id = int(pad)
        print(f"[load] prompt={tok.convert_ids_to_tokens(self._prompt_ids)} "
              f"eos={self._eos_id} pad={self._pad_id}")
        return self.model

    def preprocess(self, ex):
        audio = ex["audio"]["array"]; sr = ex["audio"]["sampling_rate"]
        if sr != 16000:
            import librosa
            audio = librosa.resample(np.asarray(audio, dtype=np.float32), orig_sr=sr, target_sr=16000)
        text = normalize_ar(ex["text"])
        return {"audio": np.asarray(audio, dtype=np.float32), "text": text,
                "audio_len": len(audio) / 16000.0}

    def _features(self, audios):
        enc = self.processor(audio=[np.asarray(a, dtype=np.float32) for a in audios],
                             language=self.lang, sampling_rate=16000)
        chunk_index = enc.pop("audio_chunk_index", None)
        return enc, chunk_index

    def collate(self, feats):
        enc, chunk_index = self._features([f["audio"] for f in feats])
        if enc["input_features"].shape[0] != len(feats):
            raise RuntimeError(
                f"feature extractor chunked {len(feats)} clips into "
                f"{enc['input_features'].shape[0]} rows -- keep training clips <= max_audio_clip_s")
        tok = self.processor.tokenizer
        P = self._prompt_ids
        seqs = [P + tok(f["text"], add_special_tokens=False)["input_ids"] + [self._eos_id]
                for f in feats]
        maxT = max(len(s) for s in seqs) - 1
        dec_in  = torch.full((len(seqs), maxT), self._pad_id, dtype=torch.long)
        labels  = torch.full((len(seqs), maxT), -100, dtype=torch.long)
        dmask   = torch.zeros((len(seqs), maxT), dtype=torch.long)
        for i, s in enumerate(seqs):
            L = len(s) - 1
            dec_in[i, :L] = torch.tensor(s[:-1], dtype=torch.long)
            dmask[i, :L] = 1
            lab = torch.tensor(s[1:], dtype=torch.long)
            lab[:len(P) - 1] = -100                      # don't train on the prompt
            labels[i, :L] = lab
        batch = {"input_features": enc["input_features"],
                 "decoder_input_ids": dec_in, "decoder_attention_mask": dmask,
                 "labels": labels}
        if "attention_mask" in enc: batch["attention_mask"] = enc["attention_mask"]
        batch["text"]   = [f["text"] for f in feats]
        batch["_audio"] = [f["audio"] for f in feats]
        return batch

    _MODEL_KEYS = ("input_features", "attention_mask", "decoder_input_ids",
                   "decoder_attention_mask", "labels")

    def train_step(self, batch) -> torch.Tensor:
        inputs = {}
        for k in self._MODEL_KEYS:
            v = batch.get(k)
            if v is None: continue
            v = v.to(DEVICE)
            if v.is_floating_point(): v = v.to(COMPUTE_DTYPE)   # input_features -> bf16
            inputs[k] = v
        return self.model(**inputs).loss

    @torch.no_grad()
    def generate(self, batch):
        enc, chunk_index = self._features(list(batch["_audio"]))
        enc = enc.to(DEVICE)
        req = {k: (v.to(COMPUTE_DTYPE) if torch.is_tensor(v) and v.is_floating_point() else v)
               for k, v in enc.items()}
        out_ids = self.model.generate(**req, max_new_tokens=256)
        gen = out_ids[:, req["decoder_input_ids"].shape[1]:]
        texts = self.processor.tokenizer.batch_decode(gen, skip_special_tokens=True)
        if chunk_index is not None and any(c[1] is not None for c in chunk_index):
            texts = self.processor._reassemble_chunk_texts(texts, chunk_index, " ")
        return [str(t).strip() for t in texts]

## Cell 7 — Registry

In [7]:
REGISTRY: Dict[str, Callable[..., ModelAdapter]] = {
    "CohereLabs/cohere-transcribe-arabic-07-2026": CohereAsrAdapter,
}

def get_adapter(name, **kw) -> ModelAdapter:
    if name not in REGISTRY: raise KeyError(f"{name} not registered. Have: {list(REGISTRY)}")
    a = REGISTRY[name](name, **kw); a.name = name; return a

## Cell 8 — Datasets

Same real North-Levantine (Palestinian) Arabic corpus + soundfile decode as the sibling
notebooks. Smoke test uses clips <= 30s.

In [8]:
from datasets import load_from_disk, Audio, Dataset
import soundfile as sf, io, random as _random

REAL_DATA_DIR = Path("/workspace/asr/Palestinian-ASR/omnilingual_selected/apc_north_levantine_all_splits")

def _materialize_audio(ds):
    def gen():
        for row in ds:
            wav, sr = sf.read(io.BytesIO(row["audio"]["bytes"]), dtype="float32")
            if wav.ndim > 1: wav = wav.mean(axis=1)
            out = dict(row); out["audio"] = {"array": wav, "sampling_rate": sr}
            yield out
    return Dataset.from_generator(gen)

def load_splits(smoke=SMOKE_TEST):
    ds = load_from_disk(str(REAL_DATA_DIR))
    if "raw_text" in ds.column_names and "text" not in ds.column_names:
        ds = ds.rename_column("raw_text", "text")
    ds = ds.cast_column("audio", Audio(decode=False))
    if smoke:
        durations = ds["duration"]
        short_idx = [i for i, d in enumerate(durations) if d <= 30.0]
        _random.Random(SEED).shuffle(short_idx)
        # 1 fake train / 1 fake val / 1 fake test sample -- real audio+text.
        splits = {"train": ds.select(short_idx[0:1]),
                  "validation": ds.select(short_idx[1:2]),
                  "test": ds.select(short_idx[2:3])}
    else:
        ds = ds.shuffle(seed=SEED)
        n = len(ds); n_tr, n_va = int(n * 0.8), int(n * 0.9)
        splits = {"train": ds.select(range(0, n_tr)),
                  "validation": ds.select(range(n_tr, n_va)),
                  "test": ds.select(range(n_va, n))}
    for k in splits: splits[k] = _materialize_audio(splits[k])
    return splits

SPLITS = load_splits()
{k: len(v) for k, v in SPLITS.items()}

{'train': 1, 'validation': 1, 'test': 1}

## Cell 9 — PredictAPI (cached)

In [9]:
def _fingerprint(ds) -> str:
    try: h = ds._fingerprint
    except Exception: h = str(len(ds))
    return hashlib.md5(f"{h}{len(ds)}".encode()).hexdigest()[:10]

class PredictAPI:
    @staticmethod
    def _path(model_name, split, ds, stage):
        slug = model_name.replace("/", "__")
        return PRED_DIR / f"{slug}__{split}__{_fingerprint(ds)}__{stage}.json"

    @staticmethod
    def run(adapter, ds, split="test", stage="base", batch_size=4, force=False):
        p = PredictAPI._path(adapter.name, split, ds, stage)
        if p.exists() and not force:
            print(f"[predict] CACHE HIT -> {p.name}"); return json.loads(p.read_text(encoding="utf-8"))
        print(f"[predict] generating ({stage}, {split}, n={len(ds)})")
        feats = [adapter.preprocess(ex) for ex in ds]
        preds, refs = [], []
        adapter.model.eval()
        for i in range(0, len(feats), batch_size):
            b = adapter.collate(feats[i:i+batch_size])
            preds.extend(adapter.generate(b)); refs.extend(b["text"])
            print(f"  {min(i+batch_size,len(feats))}/{len(feats)}", end="\r")
        rec = {"model": adapter.name, "split": split, "stage": stage,
               "predictions": preds, "references": refs, "n": len(preds), "ts": time.time()}
        p.write_text(json.dumps(rec, ensure_ascii=False, indent=2), encoding="utf-8")
        print(f"\n[predict] saved -> {p.name}")
        return rec

## Cell 10 — EvaluateAPI (cached)

In [10]:
class EvaluateAPI:
    @staticmethod
    def _path(model_name, split, stage, pred_record=None):
        slug = model_name.replace('/', '__')
        if pred_record is None:
            return METRIC_DIR / f"{slug}__{split}__{stage}.json"
        # Content-address the metric to the exact predictions it scores. PredictAPI already
        # keys on the dataset fingerprint; without the same discipline here a metric computed
        # on an older corpus gets silently re-served after the data changes, producing a wrong
        # base WER and a meaningless base->tuned delta.
        payload = json.dumps([pred_record["predictions"], pred_record["references"]],
                             ensure_ascii=False, sort_keys=True).encode("utf-8")
        h = hashlib.md5(payload).hexdigest()[:10]
        return METRIC_DIR / f"{slug}__{split}__{h}__{stage}.json"

    @staticmethod
    def run(model_name, pred_record, split="test", stage="base", force=False):
        p = EvaluateAPI._path(model_name, split, stage, pred_record)
        if p.exists() and not force:
            m = json.loads(p.read_text()); print(f"[eval] CACHE HIT -> {m}"); return m
        m = compute_wer_cer(pred_record["predictions"], pred_record["references"])
        m.update({"model": model_name, "split": split, "stage": stage})
        p.write_text(json.dumps(m, indent=2))
        print(f"[eval] WER={m['wer']:.4f} CER={m['cer']:.4f} (n={m['n']}) -> {p.name}")
        return m

## Cell 11 — Build adapter + load base model

In [11]:
set_seed()
adapter = get_adapter(MODEL_NAME, lang=LANG)
adapter.load_base()
n_params = sum(p.numel() for p in adapter.model.parameters())
print(f"{MODEL_NAME}: {n_params/1e6:.1f}M params | loss_type={adapter.loss_type}")

[load] CohereLabs/cohere-transcribe-arabic-07-2026


Loading weights:   0%|          | 0/2150 [00:00<?, ?it/s]

Loading weights:   6%|▋         | 137/2150 [00:00<00:01, 1311.60it/s]

Loading weights:  13%|█▎        | 269/2150 [00:00<00:02, 781.36it/s] 

Loading weights:  17%|█▋        | 359/2150 [00:00<00:02, 748.02it/s]

...

Loading weights:  90%|█████████ | 1935/2150 [00:02<00:00, 696.97it/s]

Loading weights:  94%|█████████▎| 2013/2150 [00:03<00:00, 707.73it/s]

Loading weights:  97%|█████████▋| 2085/2150 [00:03<00:00, 613.95it/s]

Loading weights: 100%|██████████| 2150/2150 [00:03<00:00, 657.47it/s]

[load] prompt=['▁', '<|startofcontext|>', '<|startoftranscript|>', '<|emo:undefined|>', '<|ar|>', '<|ar|>', '<|pnc|>', '<|noitn|>', '<|notimestamp|>', '<|nodiarize|>'] eos=3 pad=2
CohereLabs/cohere-transcribe-arabic-07-2026: 2065.6M params | loss_type=seq2seq


## Cell 12 — Baseline preds + eval on test (cached)

In [12]:
base_preds   = PredictAPI.run(adapter, SPLITS["test"], split="test", stage="base",
                              batch_size=ConfigAPI.train(MODEL_NAME).per_device_eval_batch_size)
base_metrics = EvaluateAPI.run(MODEL_NAME, base_preds, split="test", stage="base")
for p, r in list(zip(base_preds["predictions"], base_preds["references"]))[:3]:
    print(f"REF : {r}\nHYP : {p}\n")

[predict] CACHE HIT -> CohereLabs__cohere-transcribe-arabic-07-2026__test__15bf14064d__base.json
[eval] CACHE HIT -> {'wer': 0.36585365853658536, 'cer': 0.13901345291479822, 'n': 1, 'model': 'CohereLabs/cohere-transcribe-arabic-07-2026', 'split': 'test', 'stage': 'base'}
REF : بعدين منحط البهارات فوين وبعد ما منحط البهارات فوين منخلين يغلو ليستو شوي بعدين منشلح فوين الفريكه وبس تستوي مناكلا ولا اطيب من هيك بتاخد مده الاستوا noise لالا تاريبا حوالي شي ساعه علي الغاز مجرد ما تستوي بتصير جاهزه للاكل
HYP : بعدين بنحط البهارات فوقهم وبعد ما بنحط البهارات فوقهم بنخليهم يغلوا ليستووا شوي، بعدين بنشلح فوقهم الفريكه وبس تستوي بناكلها ولا اطيب من هيك، بتاخذ مده الاستوى لها تقريبا حوالي شي ساعه على الغاز مجرد ما تستوي بتصير جاهزه للاكل.



## Cell 13 — Apply LoRA

In [13]:
lora_spec  = ConfigAPI.lora(MODEL_NAME)
train_spec = ConfigAPI.train(MODEL_NAME)
if SMOKE_TEST:
    train_spec.num_epochs = 2
    train_spec.early_stopping_patience = 4
adapter.apply_lora(lora_spec)

[lora] discovered target_modules: ['fc1', 'fc2', 'k_proj', 'linear1', 'linear2', 'o_proj', 'q_proj', 'v_proj']


trainable params: 61,865,984 || all params: 2,127,513,856 || trainable%: 2.9079
[lora] peft


PeftModel(
  (base_model): LoraModel(
    (model): CohereAsrForConditionalGeneration(
      (model): CohereAsrModel(
        (encoder): ParakeetEncoder(
          (subsampling): ParakeetEncoderSubsamplingConv2D(
            (layers): ModuleList(
              (0): Conv2d(1, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
              (1): ReLU()
              (2): Conv2d(256, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), groups=256)
              (3): Conv2d(256, 256, kernel_size=(1, 1), stride=(1, 1))
              (4): ReLU()
              (5): Conv2d(256, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), groups=256)
              (6): Conv2d(256, 256, kernel_size=(1, 1), stride=(1, 1))
              (7): ReLU()
            )
            (linear): Linear(in_features=4096, out_features=1280, bias=True)
          )
          (encode_positions): ParakeetEncoderRelPositionalEncoding()
          (layers): ModuleList(
            (0-47): 48 x ParakeetEncoderBlock(


## Cell 14 — TrainAPI

Same custom loop as the sibling notebooks (per-epoch train loss / val loss / val WER / val CER,
early stopping on WER patience 4, best-WER checkpoint saved as a real PEFT adapter).

In [14]:
import mlflow
from contextlib import nullcontext
from torch.utils.data import DataLoader, Sampler

def _amp(spec):
    """bf16 autocast on CUDA; no-op elsewhere so the loop also runs on CPU."""
    if DEVICE == "cuda":
        return torch.autocast("cuda", dtype=torch.bfloat16)
    return nullcontext()

class _ListDS(torch.utils.data.Dataset):
    def __init__(self, feats): self.f = feats
    def __len__(self): return len(self.f)
    def __getitem__(self, i): return self.f[i]

class LengthGroupedSampler(Sampler):
    """Buckets examples into batch_size-sized windows sorted by audio length (for padding
    efficiency), then shuffles the *order of windows* every epoch -- so training still sees
    a different batch order each pass instead of a frozen short->long sweep. A fresh
    torch.randperm before the sort also jitters which items land in which window across
    epochs (tie-breaking), not just the window order."""
    def __init__(self, lengths, batch_size):
        self.lengths = lengths; self.batch_size = batch_size
    def __len__(self): return len(self.lengths)
    def __iter__(self):
        idx = torch.randperm(len(self.lengths)).tolist()
        idx.sort(key=lambda i: self.lengths[i])
        buckets = [idx[i:i + self.batch_size] for i in range(0, len(idx), self.batch_size)]
        order = torch.randperm(len(buckets)).tolist()
        out = []
        for b in order: out.extend(buckets[b])
        return iter(out)

def _rng_state():
    return {"python": random.getstate(), "numpy": np.random.get_state(),
            "torch": torch.get_rng_state(),
            "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None}

def _restore_rng(state):
    random.setstate(state["python"]); np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch"])
    if state.get("cuda") is not None and torch.cuda.is_available():
        torch.cuda.set_rng_state_all(state["cuda"])

class TrainAPI:
    @staticmethod
    def _prep(adapter, ds, spec):
        """Pre-flight: preprocess + drop over-long / empty items before the loop."""
        feats, dropped = [], 0
        for ex in ds:
            f = adapter.preprocess(ex)
            if f["audio_len"] > spec.max_audio_seconds: dropped += 1; continue
            if not f["text"].strip():                   dropped += 1; continue
            if len(f.get("labels") or []) > spec.max_label_tokens:
                f["labels"] = f["labels"][:spec.max_label_tokens]
            feats.append(f)
        print(f"[prep] kept {len(feats)}, dropped {dropped}")
        return feats

    @staticmethod
    @torch.no_grad()
    def _validate(adapter, loader, spec):
        adapter.model.eval(); losses, preds, refs = [], [], []
        for b in loader:
            g = {k: (v.to(DEVICE) if torch.is_tensor(v) else v) for k, v in b.items()}
            try:
                with _amp(spec):
                    losses.append(float(adapter.train_step(g)))
            except Exception as e:
                print(f"[val] loss skipped: {e}")
            preds.extend(adapter.generate(g)); refs.extend(b["text"])
        m = compute_wer_cer(preds, refs)
        m["val_loss"] = float(np.mean(losses)) if losses else float("nan")
        m["_preds"], m["_refs"] = preds, refs
        return m

    @staticmethod
    def run(adapter, splits, spec: TrainConfigSpec, lora_spec: LoRAConfigSpec, resume: bool = True):
        slug = adapter.name.replace("/", "__")
        run_root = CKPT_DIR / slug
        best_dir = run_root / "best"; best_dir.mkdir(parents=True, exist_ok=True)

        tr_f = TrainAPI._prep(adapter, splits["train"], spec)
        va_f = TrainAPI._prep(adapter, splits["validation"], spec)
        tr_kw = {}
        if spec.dataloader_num_workers > 0:
            tr_kw["persistent_workers"] = spec.dataloader_persistent_workers
            tr_kw["prefetch_factor"] = spec.dataloader_prefetch_factor
        tr = DataLoader(_ListDS(tr_f), batch_size=spec.per_device_train_batch_size,
                        sampler=LengthGroupedSampler([f["audio_len"] for f in tr_f],
                                                     spec.per_device_train_batch_size),
                        collate_fn=adapter.collate, num_workers=spec.dataloader_num_workers,
                        pin_memory=(spec.dataloader_pin_memory and DEVICE == "cuda"),
                        drop_last=False, **tr_kw)
        va = DataLoader(_ListDS(va_f), batch_size=spec.per_device_eval_batch_size, shuffle=False,
                        collate_fn=adapter.collate, num_workers=spec.dataloader_num_workers)

        params = adapter.trainable_parameters()
        opt = None
        if DEVICE == "cuda":            # bitsandbytes 8-bit optimizers are CUDA-only
            try:
                import bitsandbytes as bnb
                opt = bnb.optim.AdamW8bit(params, lr=spec.learning_rate, weight_decay=spec.weight_decay)
            except Exception as e:
                print(f"[opt] AdamW8bit unavailable ({e}); using torch.AdamW")
        if opt is None:
            opt = torch.optim.AdamW(params, lr=spec.learning_rate, weight_decay=spec.weight_decay)

        steps_pe = max(1, math.ceil(len(tr) / spec.gradient_accumulation_steps))
        total    = steps_pe * spec.num_epochs
        save_steps = spec.save_steps or max(200, steps_pe // 3)
        from transformers import get_linear_schedule_with_warmup
        sched = get_linear_schedule_with_warmup(opt, int(total * spec.warmup_ratio), total)

        best_wer, bad_epochs, gstep, history, start_epoch, mlflow_run_id = \
            float("inf"), 0, 0, [], 1, None

        ckpts = sorted(run_root.glob("ckpt_step*"))
        if resume and ckpts:
            last = ckpts[-1]
            try:
                state = json.loads((last / "trainer_state.json").read_text())
                adapter.load_checkpoint(last)
                # weights_only=False: these are our own trusted local checkpoint files, not
                # untrusted downloads. Needed because torch >=2.6 defaults weights_only=True,
                # which rejects the numpy-backed RNG state (numpy.random.get_state() pickles via
                # numpy's own _reconstruct, not in the default safe-globals allowlist) and can
                # also reject optimizer state depending on the optimizer's internals.
                opt.load_state_dict(torch.load(last / "optimizer.pt", map_location=DEVICE, weights_only=False))
                sched.load_state_dict(torch.load(last / "scheduler.pt", map_location=DEVICE, weights_only=False))
                _restore_rng(torch.load(last / "rng.pt", map_location="cpu", weights_only=False))
                best_wer, bad_epochs = state["best_wer"], state["bad_epochs"]
                gstep, start_epoch = state["gstep"], state["epoch"] + 1
                history, mlflow_run_id = state["history"], state.get("mlflow_run_id")
                print(f"[resume] epoch {start_epoch} gstep {gstep} best_wer {best_wer:.4f} <- {last}")
            except Exception as e:
                print(f"[resume] failed ({e}); starting fresh")

        try:
            mlflow.start_run(run_id=mlflow_run_id, run_name=f"{slug}-lora")
        except Exception as e:
            # A resumed run_id can be unusable for reasons outside our control (belongs to
            # a different active experiment -- e.g. this same checkpoint dir was previously
            # used by a sibling notebook under a different MLflow experiment name; a
            # manually deleted run; a different MLFLOW_TRACKING_URI). Never let bookkeeping
            # block real training -- fall back to a fresh run instead of crashing.
            print(f"[mlflow] could not resume run {mlflow_run_id} ({e}); starting a new run")
            mlflow_run_id = None
            mlflow.start_run(run_id=None, run_name=f"{slug}-lora")
        mlflow_run_id = mlflow.active_run().info.run_id
        try:
            params_flat = {f"train.{k}": (str(v) if isinstance(v, (list, type(None))) else v)
                           for k, v in asdict(spec).items()}
            params_flat.update({f"lora.{k}": (str(v) if isinstance(v, (list, type(None))) else v)
                                for k, v in asdict(lora_spec).items()})
            params_flat.update({"model": adapter.name, "lang": LANG, "smoke": SMOKE_TEST,
                                "save_steps": save_steps})
            mlflow.log_params(params_flat)
        except Exception as e:
            print(f"[mlflow] log_params skipped ({e})")
        try:
            mlflow.set_tags({"dataset_version": REAL_DATA_DIR.name,
                             "hardware": torch.cuda.get_device_name(0) if DEVICE == "cuda" else "cpu",
                             "stage": "dev" if SMOKE_TEST else "experiment"})
        except Exception as e:
            print(f"[mlflow] set_tags skipped ({e})")

        def _save_numbered_ckpt(epoch):
            nonlocal gstep
            d = run_root / f"ckpt_step{gstep:08d}"; d.mkdir(parents=True, exist_ok=True)
            adapter.save_checkpoint(d)
            torch.save(opt.state_dict(), d / "optimizer.pt")
            torch.save(sched.state_dict(), d / "scheduler.pt")
            torch.save(_rng_state(), d / "rng.pt")
            (d / "trainer_state.json").write_text(json.dumps(
                {"epoch": epoch, "gstep": gstep, "best_wer": best_wer, "bad_epochs": bad_epochs,
                 "history": history, "mlflow_run_id": mlflow_run_id}, indent=2))
            kept = sorted(run_root.glob("ckpt_step*"))
            for old in kept[:-spec.save_total_limit] if spec.save_total_limit > 0 else kept:
                shutil.rmtree(old, ignore_errors=True)
            return d

        for epoch in range(start_epoch, spec.num_epochs + 1):
            t_epoch = time.time()
            adapter.model.train(); ep_loss, nb = 0.0, 0
            opt.zero_grad(set_to_none=True)
            for i, b in enumerate(tr):
                t_step = time.time()
                g = {k: (v.to(DEVICE) if torch.is_tensor(v) else v) for k, v in b.items()}
                with _amp(spec):
                    loss = adapter.train_step(g) / spec.gradient_accumulation_steps
                if not torch.isfinite(loss):
                    print(f"[nan] step {i} non-finite loss, skipping batch")
                    opt.zero_grad(set_to_none=True); continue
                loss.backward()
                if (i + 1) % spec.gradient_accumulation_steps == 0 or (i + 1) == len(tr):
                    gnorm = torch.nn.utils.clip_grad_norm_(params, spec.max_grad_norm)
                    if not torch.isfinite(gnorm):
                        print(f"[nan] step {i} non-finite grad norm, skipping update")
                        opt.zero_grad(set_to_none=True); continue
                    opt.step(); sched.step(); opt.zero_grad(set_to_none=True); gstep += 1
                    mem = torch.cuda.memory_allocated() / 1e9 if DEVICE == "cuda" else 0.0
                    mlflow.log_metrics({"train/step_loss": float(loss) * spec.gradient_accumulation_steps,
                                        "train/grad_norm": float(gnorm),
                                        "train/lr": sched.get_last_lr()[0],
                                        "train/step_time_s": time.time() - t_step,
                                        "train/gpu_mem_gb": mem}, step=gstep)
                    if gstep % save_steps == 0:
                        _save_numbered_ckpt(epoch)
                ep_loss += float(loss) * spec.gradient_accumulation_steps; nb += 1

            train_loss = ep_loss / max(nb, 1)
            vm = TrainAPI._validate(adapter, va, spec)
            row = {"epoch": epoch, "train_loss": train_loss, "val_loss": vm["val_loss"],
                   "val_wer": vm["wer"], "val_cer": vm["cer"]}
            history.append(row)
            throughput = sum(f["audio_len"] for f in tr_f) / max(time.time() - t_epoch, 1e-6)
            mlflow.log_metrics({"epoch": epoch, "train/loss": train_loss, "val/loss": vm["val_loss"],
                                "val/wer": vm["wer"], "val/cer": vm["cer"],
                                "train/throughput_audio_s_per_s": throughput,
                                "train/epoch_time_s": time.time() - t_epoch}, step=gstep)
            try:
                import pandas as pd
                qual = pd.DataFrame({"reference": vm["_refs"][:5], "hypothesis": vm["_preds"][:5]})
                mlflow.log_table(qual, artifact_file=f"qualitative/epoch_{epoch:03d}.json")
            except Exception as e:
                print(f"[mlflow] qualitative table skipped ({e})")
            print(f"epoch {epoch:>3} | train {train_loss:.4f} | val {vm['val_loss']:.4f} "
                  f"| WER {vm['wer']:.4f} | CER {vm['cer']:.4f}")

            # ---- best-WER checkpoint (protected) + early stopping ----
            if vm["wer"] < best_wer - 1e-6:
                best_wer, bad_epochs = vm["wer"], 0
                adapter.save_checkpoint(best_dir)
                (best_dir / "best.json").write_text(json.dumps({**row, "gstep": gstep}, indent=2))
                print(f"  -> new best WER {best_wer:.4f}, saved to {best_dir}")
            else:
                bad_epochs += 1
                print(f"  -> no improvement ({bad_epochs}/{spec.early_stopping_patience})")

            # ---- epoch-boundary checkpoint, regardless of step count: clean resume fallback ----
            _save_numbered_ckpt(epoch)

            if bad_epochs >= spec.early_stopping_patience:
                print(f"[early-stop] epoch {epoch}, best WER {best_wer:.4f}"); break

        mlflow.log_metric("best_val_wer", best_wer, step=gstep)
        (run_root / "history.json").write_text(json.dumps(history, indent=2))
        return {"best_wer": best_wer, "best_dir": str(best_dir), "history": history,
                "mlflow_run_id": mlflow_run_id}


## Cell 15 — Train

In [15]:
set_seed()
train_out = TrainAPI.run(adapter, SPLITS, train_spec, lora_spec)
print(f"best val WER: {train_out['best_wer']:.4f} @ {train_out['best_dir']}")

[prep] kept 1, dropped 0


[prep] kept 1, dropped 0


epoch   1 | train 11.0661 | val 17.2729 | WER 0.2791 | CER 0.1164


  -> new best WER 0.2791, saved to /workspace/asr_env/checkpoints/CohereLabs__cohere-transcribe-arabic-07-2026/best


epoch   2 | train 10.9856 | val 18.7727 | WER 0.3256 | CER 0.1422
  -> no improvement (1/4)


best val WER: 0.2791 @ /workspace/asr_env/checkpoints/CohereLabs__cohere-transcribe-arabic-07-2026/best


## Cell 16 — Load best checkpoint, predict + evaluate on test, save

In [16]:
best = Path(train_out["best_dir"])
try:
    adapter.load_checkpoint(best)
    print(f"[ckpt] restored best adapter <- {best}")
except Exception as e:
    print(f"[ckpt] restore failed ({e})")

tuned_preds   = PredictAPI.run(adapter, SPLITS["test"], split="test", stage="tuned",
                               batch_size=train_spec.per_device_eval_batch_size, force=True)
tuned_metrics = EvaluateAPI.run(MODEL_NAME, tuned_preds, split="test", stage="tuned", force=True)

dataset_hours  = {k: sum(v["duration"]) / 3600.0 for k, v in SPLITS.items()}
dataset_counts = {k: len(v) for k, v in SPLITS.items()}

summary = {
    "model": MODEL_NAME, "lang": LANG, "smoke_test": SMOKE_TEST,
    "base":  {"wer": base_metrics["wer"],  "cer": base_metrics["cer"]},
    "tuned": {"wer": tuned_metrics["wer"], "cer": tuned_metrics["cer"]},
    "delta": {"wer": base_metrics["wer"] - tuned_metrics["wer"],
              "cer": base_metrics["cer"] - tuned_metrics["cer"]},
    "best_val_wer": train_out["best_wer"],
    "dataset_counts": dataset_counts, "dataset_hours": dataset_hours,
    "lora": asdict(lora_spec), "train": asdict(train_spec),
}
sp = METRIC_DIR / f"{MODEL_NAME.replace('/','__')}__SUMMARY.json"
sp.write_text(json.dumps(summary, indent=2, ensure_ascii=False))

mlflow.log_metrics({"test/base_wer": base_metrics["wer"],   "test/base_cer": base_metrics["cer"],
                    "test/tuned_wer": tuned_metrics["wer"], "test/tuned_cer": tuned_metrics["cer"],
                    **{f"data/{k}_hours": v for k, v in dataset_hours.items()},
                    **{f"data/{k}_count": v for k, v in dataset_counts.items()}})
try:
    mlflow.log_artifact(str(sp), artifact_path="summary")
    import pandas as pd
    mlflow.log_table(pd.DataFrame({"reference": tuned_preds["references"],
                                   "base_hyp": base_preds["predictions"],
                                   "tuned_hyp": tuned_preds["predictions"]}),
                     artifact_file="qualitative/test_predictions.json")
except Exception as e:
    print(f"[mlflow] artifact logging skipped ({e})")
mlflow.end_run()
print(json.dumps(summary, indent=2, ensure_ascii=False))


[ckpt] restored best adapter <- /workspace/asr_env/checkpoints/CohereLabs__cohere-transcribe-arabic-07-2026/best
[predict] generating (tuned, test, n=1)


  1/1
[predict] saved -> CohereLabs__cohere-transcribe-arabic-07-2026__test__15bf14064d__tuned.json
[eval] WER=0.3659 CER=0.1390 (n=1) -> CohereLabs__cohere-transcribe-arabic-07-2026__test__bd8b750aab__tuned.json


{
  "model": "CohereLabs/cohere-transcribe-arabic-07-2026",
  "lang": "ar",
  "smoke_test": true,
  "base": {
    "wer": 0.36585365853658536,
    "cer": 0.13901345291479822
  },
  "tuned": {
    "wer": 0.36585365853658536,
    "cer": 0.13901345291479822
  },
  "delta": {
    "wer": 0.0,
    "cer": 0.0
  },
  "best_val_wer": 0.27906976744186046,
  "dataset_counts": {
    "train": 1,
    "validation": 1,
    "test": 1
  },
  "dataset_hours": {
    "train": 0.008036799768518519,
    "validation": 0.0077747627314814815,
    "test": 0.006103252314814815
  },
  "lora": {
    "r": 32,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "bias": "none",
    "target_modules": null,
    "modules_to_save": null,
    "task_type": null
  },
  "train": {
    "num_epochs": 2,
    "early_stopping_patience": 4,
    "metric_for_best": "wer",
    "greater_is_better": false,
    "per_device_train_batch_size": 1,
    "per_device_eval_batch_size": 2,
    "gradient_accumulation_steps": 8,
    "learning_rate":

## Cell 17 — One-shot smoke wrapper

Same code path end-to-end (load → base predict/eval → LoRA → train → tuned predict/eval),
callable for any registered CohereAsr checkpoint.

In [17]:
def smoke(model_name, splits):
    set_seed()
    a = get_adapter(model_name, lang=LANG); a.load_base()
    bp = PredictAPI.run(a, splits["test"], "test", "base", batch_size=2)
    bm = EvaluateAPI.run(model_name, bp, "test", "base")
    ls, ts = ConfigAPI.lora(model_name), ConfigAPI.train(model_name)
    ts.num_epochs = 2; ts.per_device_train_batch_size = 1; ts.gradient_accumulation_steps = 2
    a.apply_lora(ls)
    out = TrainAPI.run(a, splits, ts, ls)
    tp = PredictAPI.run(a, splits["test"], "test", "tuned", batch_size=2, force=True)
    tm = EvaluateAPI.run(model_name, tp, "test", "tuned", force=True)
    if mlflow.active_run() is not None:   # TrainAPI.run() started one; this loop never reaches
        mlflow.end_run()                  # the Cell 16 save-results cell that normally ends it
    del a.model, a; gc.collect(); torch.cuda.empty_cache()
    return {"model": model_name, "base_wer": bm["wer"], "tuned_wer": tm["wer"],
            "best_val_wer": out["best_wer"], "status": "PASS"}

results = []
for m in ["CohereLabs/cohere-transcribe-arabic-07-2026"]:
    try:
        results.append(smoke(m, SPLITS))
    except Exception as e:
        import traceback; traceback.print_exc()
        results.append({"model": m, "status": f"FAIL: {e}"})
    print("=" * 70)

import pandas as pd
pd.DataFrame(results)

[load] CohereLabs/cohere-transcribe-arabic-07-2026


Loading weights:   0%|          | 0/2150 [00:00<?, ?it/s]

Loading weights:  21%|██        | 454/2150 [00:00<00:00, 4538.98it/s]

Loading weights:  42%|████▏     | 908/2150 [00:00<00:00, 3743.97it/s]

Loading weights:  60%|██████    | 1292/2150 [00:00<00:00, 3500.83it/s]

...

[prep] kept 1, dropped 0


[resume] epoch 3 gstep 2 best_wer 0.2791 <- /workspace/asr_env/checkpoints/CohereLabs__cohere-transcribe-arabic-07-2026/ckpt_step00000002
[mlflow] log_params skipped (Changing param values is not allowed. Params were already logged='[{'key': 'train.gradient_accumulation_steps', 'old_value': '8', 'new_value': '2'}]' for run ID='fca49ad3b0bd488b9b1a98f9cfb35a04'.)


[predict] generating (tuned, test, n=1)


  1/1
[predict] saved -> CohereLabs__cohere-transcribe-arabic-07-2026__test__15bf14064d__tuned.json
[eval] WER=0.3659 CER=0.1390 (n=1) -> CohereLabs__cohere-transcribe-arabic-07-2026__test__bd8b750aab__tuned.json


,model,base_wer,tuned_wer,best_val_wer,status
0,CohereLabs/cohere-transcribe-arabic-07-2026,0.365854,0.365854,0.27907,PASS
